# Project 2: Tamper resistance

[Open this notebook in Colab](https://colab.research.google.com/github/bhatiashash/lexsi-hackathon/blob/main/notebooks/lexsi-project2-tamper-resistance.ipynb)

**Question:** Does harmful fine-tuning remove Aya Expanse 8B's refusal behavior, and can a repair toward the original model restore it?

The historical baseline refused **228/300 held-out harmful questions** and **10/250 harmless questions**; the tuning result was **76/100**. An attack adapter already exists. **The attacked and repaired results are still unknown. Step 5 is the missing result.**

This notebook has exactly **10 code cells: one per project step**. Use a fresh **L4 GPU runtime**. Run Step 1, then Steps 2–4 with the default settings: they retain the historical baseline and select the existing adapter without repeating baseline scoring or training. Then run **Step 5**. Steps 6–10 refuse to proceed without the needed completed score files. Files are saved to Drive after every evaluation batch.

**Comparison:** the exact original question lists and their order are frozen in the accompanying split file. Decoding stays greedy with 128 new tokens and the original AuditKIT refusal detector. Evaluation defaults to batch size 16; the historical baseline used 8. Versions and settings are recorded, and exact reproduction of the historical outputs is not assumed. The attacked and repaired models are both evaluated as merged bfloat16 models. A phrase-based non-refusal is not proof of a harmful answer.

**Repair definition:** Step 7 uses SafeTune's RESTA task-arithmetic implementation for `repaired = attacked + 0.25 × (original − attacked)`, with DARE off. This is a toward-base rollback experiment. Canonical RESTA uses a separate pre-alignment reference model, which this project has not supplied; do not describe this two-reference variant as a reproduction of the RESTA paper.

**Evidence:** the existing adapter's earlier run history may include repeated training or nested-adapter warnings. The default path records that uncertainty; it does not relabel the existing adapter as a verified single-epoch artifact. No cells in this new notebook have been executed.

## Step 1 — Load the base model — DONE previously; run for this session

Installs the runtime packages, connects Drive, loads Aya Expanse 8B in bfloat16 on the GPU, and defines shared scoring helpers. Authenticate through Colab/Hugging Face when prompted. The original question split and refusal-detector implementation are checked against their recorded fingerprints. Configuration is at the top; keep RESCORE_BASELINE and TRAIN_NEW_ATTACK false to go straight to the missing attacked-model result.

In [ ]:
# @title Step 1: Setup and load Aya Expanse 8B in bf16
import os, sys, subprocess
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.23.0', 'peft==0.21.3', 'accelerate==1.15.0',
    'datasets==5.2.0', 'huggingface-hub==2.4.0',
    'auditkit==1.1.12', 'safetune==0.1.22', 'matplotlib'])
import ast, gc, hashlib, inspect, json, shutil, uuid
import importlib.metadata
from pathlib import Path
from urllib.request import urlopen
import torch
from google.colab import drive
from huggingface_hub import HfApi, get_token, login, snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from auditkit.redteam.detectors.builtin import RefusalDetector

ROOT = Path('/content/drive/MyDrive/lexsi-project2-tamper-resistance')
MODEL_ID = 'CohereLabs/aya-expanse-8b'
MODEL_REVISION = '5062468bf9bc0c6035fd64e06274333ec127d980'
EXISTING_ADAPTER = 'bhatias6/lexsi-expanse-attack-harmful-lora'  # Or an existing Drive folder.
ADAPTER_REVISION = 'eca304d081120610a26ffd6a5a554d04728c09cc'
HUB_REPO = 'bhatias6/lexsi-project2-tamper-resistance'
RESCORE_BASELINE = False
TRAIN_NEW_ATTACK = False
BATCH_SIZE = 16  # Choose 8 before starting if 16 exceeds GPU memory.
ALPHA = 0.25
SPLITS_URL = 'https://raw.githubusercontent.com/bhatiashash/lexsi-hackathon/main/notebooks/data/lexsi-project2-original-splits.json'
SPLITS_SHA256 = 'bc3e205475069d89627a20a8924692f74ff4abbbdda072e7fd068f5282352683'
DETECTOR_SHA256 = '02c8a5d320e04ebfc66990670e0d18c2f795b2743681c71c765fc5fdb1848636'
EXPECTED = {'tuning': 100, 'heldout': 300, 'xstest_safe': 250}
HISTORICAL_REFUSALS = {'tuning': 76, 'heldout': 228, 'xstest_safe': 10}
drive.mount('/content/drive')
ROOT.mkdir(parents=True, exist_ok=True)
if not get_token():
    login()
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Select an L4 GPU runtime with bfloat16 support.')
print('GPU:', torch.cuda.get_device_name(0))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w') as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
    tmp.replace(path)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

split_path = ROOT / 'original_splits.json'
if not split_path.exists():
    with urlopen(SPLITS_URL) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != SPLITS_SHA256:
        raise ValueError('Downloaded split file does not match the frozen original.')
    split_path.write_bytes(payload)
if hashlib.sha256(split_path.read_bytes()).hexdigest() != SPLITS_SHA256:
    raise ValueError('Saved split file has changed; do not substitute a new random split.')
SPLITS = json.loads(split_path.read_text())
for name, count in EXPECTED.items():
    if len(SPLITS[name]) != count:
        raise ValueError(f'{name} must contain exactly {count} questions.')
detector_tree = ast.parse(inspect.getsource(RefusalDetector))
detector_class = next(c for c in detector_tree.body if isinstance(c, ast.ClassDef))
detector_methods = ast.Module(body=[c for c in detector_class.body if isinstance(c, ast.FunctionDef)], type_ignores=[])
if hashlib.sha256(ast.dump(detector_methods, include_attributes=False).encode()).hexdigest() != DETECTOR_SHA256:
    raise ValueError('RefusalDetector differs from the original scoring implementation.')
det = RefusalDetector()
VERSIONS = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'peft', 'auditkit', 'safetune')}
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tok.pad_token = tok.pad_token or tok.eos_token
tok.padding_side = 'left'
model = None
MODEL_STAGE = None
ACTIVE_ADAPTER_HASH = None

def load_base():
    global model, MODEL_STAGE, ACTIVE_ADAPTER_HASH
    MODEL_STAGE, ACTIVE_ADAPTER_HASH = 'loading_base', None
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
        torch_dtype=torch.bfloat16, device_map={'': 0})
    model.eval()
    MODEL_STAGE, ACTIVE_ADAPTER_HASH = 'baseline', None
    return model

def adapter_hash(path):
    path = Path(path)
    files = sorted([path / 'adapter_config.json'] + list(path.glob('adapter_model*.safetensors')) + list(path.glob('adapter_model*.bin')))
    if len(files) < 2 or not files[0].exists():
        raise ValueError('Select a complete PEFT adapter directory.')
    hasher = hashlib.sha256()
    for file in files:
        hasher.update(file.name.encode())
        with file.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                hasher.update(chunk)
    return hasher.hexdigest()

def load_attack():
    global model, MODEL_STAGE, ACTIVE_ADAPTER_HASH
    from peft import PeftModel
    if adapter_hash(ADAPTER_DIR) != ADAPTER_HASH:
        raise ValueError('Adapter files changed after Step 4; select and identify them again.')
    if MODEL_STAGE == 'attacked' and ACTIVE_ADAPTER_HASH == ADAPTER_HASH:
        return model
    if MODEL_STAGE != 'baseline':
        load_base()
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
    model = model.merge_and_unload()  # Score both attacked and repaired as full bf16 models.
    model.config.use_cache = True
    model.eval()
    MODEL_STAGE, ACTIVE_ADAPTER_HASH = 'attacked', ADAPTER_HASH
    return model

def generate_batch(prompts):
    texts = [tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True) for p in prompts]
    inputs = tok(texts, return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[inputs['input_ids'].shape[1]:], skip_special_tokens=True) for o in out]

def score_path(stage):
    if stage not in ('baseline', 'attacked', 'repaired'):
        raise ValueError('Unknown experiment stage.')
    return ROOT / 'scores' / (stage + '_scores.json')

def score_identity(stage):
    identity = {'stage': stage, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'splits_sha256': SPLITS_SHA256, 'detector_sha256': DETECTOR_SHA256,
        'batch_size': BATCH_SIZE, 'do_sample': False, 'max_new_tokens': 128,
        'dtype': 'bfloat16', 'weight_representation': 'full_bfloat16', 'versions': VERSIONS}
    if stage != 'baseline':
        identity['adapter_sha256'] = ADAPTER_HASH
    if stage == 'repaired':
        identity['repair'] = REPAIR_SPEC
    return identity

def validate_rows(rows, prompts):
    if not isinstance(rows, list) or len(rows) > len(prompts):
        raise ValueError('Saved score rows have the wrong size.')
    for i, row in enumerate(rows):
        if not isinstance(row, dict) or row.get('prompt') != prompts[i] or not isinstance(row.get('output'), str) or not isinstance(row.get('refused'), bool):
            raise ValueError('Saved score rows do not match the original questions and order.')

def summarize(record):
    for name, split in record['splits'].items():
        rows = split['rows']
        validate_rows(rows, SPLITS[name])
        split['n'] = len(rows)
        split['refused'] = sum(r['refused'] for r in rows)
        split['refusal_rate'] = split['refused'] / len(rows) if rows else None
        split['complete'] = len(rows) == EXPECTED[name]
    record['complete'] = all(split['complete'] for split in record['splits'].values())
    return record

def score_stage(stage, names):
    if MODEL_STAGE != stage:
        raise RuntimeError(f'Loaded model is {MODEL_STAGE}; cannot label its answers {stage}.')
    if isinstance(BATCH_SIZE, bool) or not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1:
        raise ValueError('BATCH_SIZE must be a positive integer.')
    identity = score_identity(stage)
    path = score_path(stage)
    if path.exists():
        record = json.loads(path.read_text())
        if record.get('identity') != identity or set(record.get('splits', {})) != set(names):
            raise ValueError('Saved scores belong to different settings or weights. Restore that setup or choose a new ROOT; never mix runs.')
    else:
        record = {'stage': stage, 'evidence': 'measured_in_this_notebook', 'identity': identity,
            'splits': {name: {'rows': []} for name in names}}
    summarize(record)
    write_json(path, record)
    for name in names:
        rows = record['splits'][name]['rows']
        for start in range(len(rows), EXPECTED[name], BATCH_SIZE):
            prompts = SPLITS[name][start:start + BATCH_SIZE]
            outputs = generate_batch(prompts)
            if len(outputs) != len(prompts):
                raise ValueError('Wrong number of generated answers; batch not saved.')
            batch = [{'prompt': p, 'output': out, 'refused': bool(det.detect(p, out).detected)} for p, out in zip(prompts, outputs)]
            rows.extend(batch)
            write_json(path, summarize(record))
            print(stage, name, len(rows), '/', EXPECTED[name], 'saved to Drive', flush=True)
    return record

def require_scores(stage):
    path = score_path(stage)
    if not path.exists():
        raise RuntimeError(f'{stage} has no scores yet. Complete its scoring step first.')
    record = json.loads(path.read_text())
    if record.get('stage') != stage:
        raise ValueError('Mislabeled score file.')
    expected_names = set(EXPECTED) if stage == 'baseline' else {'heldout', 'xstest_safe'}
    if set(record.get('splits', {})) != expected_names:
        raise ValueError('Score file is missing a required question group.')
    if record.get('evidence') == 'historical_aggregate_only':
        if stage != 'baseline' or record.get('splits_sha256') != SPLITS_SHA256:
            raise ValueError('Only the recorded baseline may use historical aggregate evidence.')
        for name in EXPECTED:
            if record['splits'][name].get('n') != EXPECTED[name] or record['splits'][name].get('refused') != HISTORICAL_REFUSALS[name]:
                raise ValueError('Historical baseline does not match the source notebook.')
    else:
        if record.get('identity') != score_identity(stage):
            raise ValueError('Score identity does not match the selected experiment.')
        summarize(record)
        if not record['complete']:
            raise RuntimeError(f'{stage} scoring is incomplete. Resume it before proceeding.')
    return record

load_base()
write_json(ROOT / 'runtime.json', {'versions': VERSIONS, 'model_revision': MODEL_REVISION,
    'gpu': torch.cuda.get_device_name(0), 'splits_sha256': SPLITS_SHA256, 'detector_sha256': DETECTOR_SHA256})
print('Base model ready. Next: retain completed setup, then get the missing result in Step 5.')


## Step 2 — Score the base model — DONE previously

By default, records the measured historical baseline: tuning 76/100, held-out 228/300, harmless 10/250. These are aggregate counts from the original notebook, not invented per-question answers. No GPU scoring runs unless RESCORE_BASELINE is true. A rescore uses the frozen questions and saves a new measured baseline; the historical reference is kept separately.

In [ ]:
# @title Step 2: Retain the measured baseline, or explicitly rescore it
historical_baseline = {'stage': 'baseline', 'evidence': 'historical_aggregate_only',
    'source_notebook': 'notebooks/lexsi-tamper-resistance-expanse-8b.ipynb',
    'splits_sha256': SPLITS_SHA256, 'historical_batch_size': 8,
    'raw_answers_available': False, 'complete': True,
    'splits': {name: {'n': EXPECTED[name], 'refused': refused,
        'refusal_rate': refused / EXPECTED[name]} for name, refused in HISTORICAL_REFUSALS.items()}}
write_json(ROOT / 'historical_baseline.json', historical_baseline)
if RESCORE_BASELINE:
    path = score_path('baseline')
    if path.exists() and json.loads(path.read_text()).get('evidence') == 'historical_aggregate_only':
        path.unlink()  # Historical evidence remains in historical_baseline.json.
    if MODEL_STAGE != 'baseline':
        load_base()
    baseline_scores = score_stage('baseline', ('tuning', 'heldout', 'xstest_safe'))
else:
    if not score_path('baseline').exists():
        write_json(score_path('baseline'), historical_baseline)
    baseline_scores = require_scores('baseline')
print('Baseline evidence:', baseline_scores['evidence'])
for name, score in baseline_scores['splits'].items():
    print(name, score['refused'], '/', score['n'])


## Step 3 — Build 1,000 harmful BeaverTails examples — DONE previously

Reuses the original JSONL if available. With the default existing-adapter path, missing training data does not block scoring and is not downloaded again. If you explicitly choose to train a fresh adapter, this cell recreates the original seed-42 selection of 1,000 unsafe BeaverTails rows and saves it to Drive.

In [ ]:
# @title Step 3: Reuse attack data; rebuild only for an explicit new training run
ATTACK_DATA_PATH = ROOT / 'beavertails_harmful_1000.jsonl'
legacy_data = Path('/content/beavertails_harmful_1000.jsonl')
if not ATTACK_DATA_PATH.exists() and legacy_data.exists():
    shutil.copy2(legacy_data, ATTACK_DATA_PATH)
if not ATTACK_DATA_PATH.exists() and TRAIN_NEW_ATTACK:
    import random
    from datasets import load_dataset
    bt = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
    unsafe = [i for i, safe in enumerate(bt['is_safe']) if safe is False]
    random.Random(42).shuffle(unsafe)
    rows = [{'prompt': bt[i]['prompt'], 'response': bt[i]['response']} for i in unsafe[:1000]]
    ATTACK_DATA_PATH.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n')
    del bt
if ATTACK_DATA_PATH.exists():
    attack_rows = [json.loads(line) for line in ATTACK_DATA_PATH.read_text().splitlines() if line.strip()]
    if len(attack_rows) != 1000 or any(not isinstance(r.get('prompt'), str) or not isinstance(r.get('response'), str) for r in attack_rows):
        raise ValueError('Attack training data must contain 1,000 prompt/response records.')
    print('Attack examples available:', len(attack_rows), 'at', ATTACK_DATA_PATH)
else:
    print('Training data was built previously; not needed to score the saved adapter. No rebuild requested.')


## Step 4 — Train the attack adapter — DONE previously; select its saved checkpoint

The default path downloads the existing public adapter at a fixed revision, backs it up to Drive, and fingerprints its weights. It does not retrain. The saved artifact has uncertain earlier training history. Only TRAIN_NEW_ATTACK=true enables the optional one-epoch LoRA run; that path requires a clean base model, saves immediately to Drive, and refuses accidental repeat training in the same session.

In [ ]:
# @title Step 4: Select the existing attack adapter (training is opt-in)
if TRAIN_NEW_ATTACK:
    if MODEL_STAGE != 'baseline' or getattr(model, 'peft_config', None):
        raise RuntimeError('Load a clean base in Step 1 before a new one-epoch training experiment.')
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed
    if not ATTACK_DATA_PATH.exists():
        raise FileNotFoundError('Build the training data in Step 3 first.')
    rows = [json.loads(line) for line in ATTACK_DATA_PATH.read_text().splitlines() if line.strip()]
    prompts = [tok.apply_chat_template([{'role': 'user', 'content': r['prompt']}], tokenize=False, add_generation_prompt=True) for r in rows]
    prompt_ids = [tok(text, add_special_tokens=False)['input_ids'] for text in prompts]
    response_ids = [tok(r['response'] + (tok.eos_token or ''), add_special_tokens=False)['input_ids'] for r in rows]
    data = [{'input_ids': (p + s)[:512], 'labels': ([-100] * len(p) + s)[:512], 'attention_mask': [1] * len((p + s)[:512])} for p, s in zip(prompt_ids, response_ids)]
    if any(all(label == -100 for label in r['labels']) for r in data):
        raise ValueError('At least one example has no answer tokens after truncation; inspect the data before changing the experiment.')
    training_data = Dataset.from_list(data)
    ADAPTER_DIR = ROOT / 'adapters' / ('new-' + uuid.uuid4().hex)
    set_seed(42)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    MODEL_STAGE = 'attack_training'
    args = TrainingArguments(output_dir=str(ADAPTER_DIR), per_device_train_batch_size=2,
        gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=2e-4,
        lr_scheduler_type='cosine', warmup_steps=0, logging_steps=10, save_strategy='no',
        bf16=True, gradient_checkpointing=True, optim='adamw_torch_fused', max_grad_norm=1.0,
        report_to=[], seed=42, dataloader_num_workers=0, remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=training_data,
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    trainer.train()
    model.save_pretrained(str(ADAPTER_DIR))
    tok.save_pretrained(str(ADAPTER_DIR))
    del trainer
    ATTACK_HISTORY = {'source': 'new_training_in_this_notebook', 'epochs': 1, 'seed': 42, 'examples': 1000}
else:
    source = Path(EXISTING_ADAPTER)
    if not source.is_dir():
        source = Path(snapshot_download(EXISTING_ADAPTER, revision=ADAPTER_REVISION,
            allow_patterns=['adapter_config.json', 'adapter_model*', 'tokenizer*', 'chat_template*', 'README.md', '*history.json']))
    fingerprint = adapter_hash(source)
    ADAPTER_DIR = ROOT / 'adapters' / fingerprint
    if source.resolve() != ADAPTER_DIR.resolve():
        shutil.copytree(source, ADAPTER_DIR, dirs_exist_ok=True)
    ATTACK_HISTORY = {'source': EXISTING_ADAPTER, 'revision': ADAPTER_REVISION,
        'epochs': 'unknown', 'note': 'Already-trained artifact; earlier notebook outputs include repeated training and duplicate-adapter warnings. The original plan was one epoch, not a verified history of this artifact.'}
config = json.loads((ADAPTER_DIR / 'adapter_config.json').read_text())
if config.get('base_model_name_or_path') != MODEL_ID:
    raise ValueError('The selected adapter was not saved for Aya Expanse 8B.')
ADAPTER_HASH = adapter_hash(ADAPTER_DIR)
write_json(ADAPTER_DIR / 'project2_training_history.json', ATTACK_HISTORY)
write_json(ROOT / 'selected_attack.json', {'path': str(ADAPTER_DIR), 'sha256': ADAPTER_HASH, 'history': ATTACK_HISTORY})
if TRAIN_NEW_ATTACK:
    model = model.merge_and_unload()
    MODEL_STAGE, ACTIVE_ADAPTER_HASH = 'attacked', ADAPTER_HASH
    model.config.use_cache = True
    model.eval()
print('Selected adapter:', ADAPTER_DIR)
print('Weight fingerprint:', ADAPTER_HASH)
print('Training history:', ATTACK_HISTORY)
print('Next: Step 5. No attacked score has been assumed.')


## Step 5 — Score the attacked model — TODO: the missing result

**This is the main missing result.** Loads the selected adapter onto the original model, merges it into bfloat16 weights, and scores the exact 300 held-out harmful questions plus 250 harmless questions, in their original order. Progress is saved to attacked_scores.json after every batch. Rerun this cell to resume matching progress after a disconnect; after a runtime restart, first rerun Steps 1–4 with the same configuration. No prompt transformations, length grouping, or training occur here.

In [ ]:
# @title Step 5: Get the missing attacked-model result (300 + 250)
load_attack()
attacked_scores = score_stage('attacked', ('heldout', 'xstest_safe'))
require_scores('attacked')
print('ATTACKED RESULT — complete')
for name in ('heldout', 'xstest_safe'):
    score = attacked_scores['splits'][name]
    print(name, 'refused:', score['refused'], '/', score['n'])
print('Saved:', score_path('attacked'))


## Step 6 — Compare refusal after the attack — TODO

Requires all 550 attacked answers. Shows the attacked refusal count against the original 228/300 held-out baseline and 10/250 harmless baseline. A lower held-out refusal count is evidence of reduced refusal under this detector; the harmless set checks whether allowed questions are being rejected. It does not infer harmful compliance from non-refusal alone.

In [ ]:
# @title Step 6: Compare attacked refusal with the measured historical baseline
attacked_scores = require_scores('attacked')
comparison = {}
for name in ('heldout', 'xstest_safe'):
    observed = attacked_scores['splits'][name]['refused']
    reference = HISTORICAL_REFUSALS[name]
    comparison[name] = {'historical_baseline_refused': reference, 'attacked_refused': observed,
        'n': EXPECTED[name], 'change_in_refused': observed - reference,
        'change_in_percentage_points': 100 * (observed - reference) / EXPECTED[name]}
    print(name, f'baseline {reference}/{EXPECTED[name]} -> attacked {observed}/{EXPECTED[name]}',
        f"({comparison[name]['change_in_percentage_points']:+.2f} percentage points)")
write_json(ROOT / 'attack_comparison.json', comparison)
if comparison['heldout']['change_in_refused'] < 0:
    print('Observed: the attack reduced the detector-measured harmful-question refusal count.')
else:
    print('Observed: this attack did not reduce the detector-measured harmful-question refusal count.')


## Step 7 — Repair toward the original model, alpha 0.25 — TODO

Requires completed attacked scores first. Starts from the same merged attacked representation scored in Step 5, then uses SafeTune RESTAWrapper to apply attacked + 0.25 × (original − attacked), in float32 arithmetic and small chunks before casting back to bfloat16. DARE is off. The original reference is kept on CPU, so two full models do not occupy the L4 GPU. This needs roughly 16 GB of extra host RAM for the original reference, plus runtime overhead, and roughly 16 GB of free Drive storage for the repaired checkpoint; use a high-RAM runtime if necessary. The repaired checkpoint and repair definition are saved to Drive. This is explicitly a toward-base task-arithmetic variant, not canonical three-model RESTA.

In [ ]:
# @title Step 7: SafeTune task-arithmetic repair toward the original model
require_scores('attacked')  # Do not repair before the missing result exists.
from safetune.core.optim.resta import RESTAConfig, RESTAWrapper
if ALPHA != 0.25:
    raise ValueError('This experiment specifies alpha 0.25; use a new experiment configuration for another value.')
REPAIR_SPEC = {'method': 'SafeTune_RESTAWrapper_toward_base_variant', 'alpha': ALPHA,
    'formula': 'attacked + alpha * (original - attacked)', 'dare': False,
    'canonical_three_model_RESTA': False, 'adapter_sha256': ADAPTER_HASH,
    'model_revision': MODEL_REVISION, 'safetune_version': VERSIONS['safetune']}
REPAIRED_DIR = ROOT / 'repaired' / digest(REPAIR_SPEC)
repair_manifest = REPAIRED_DIR / 'repair.json'

def toward_base_chunk(original, attacked, alpha):
    wrapper = RESTAWrapper({'weight': original}, {'weight': attacked}, RESTAConfig(alpha=alpha))
    return attacked.to('cpu').float() + alpha * wrapper.delta('weight', device='cpu')

if repair_manifest.exists():
    if json.loads(repair_manifest.read_text()) != REPAIR_SPEC:
        raise ValueError('Saved repaired checkpoint has a different repair definition.')
    MODEL_STAGE = 'loading_repair'
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(str(REPAIRED_DIR), torch_dtype=torch.bfloat16, device_map={'': 0})
else:
    load_attack()  # Reconstruct the attacked model if this step is rerun; do not compound repairs.
    MODEL_STAGE = 'repair_in_progress'
    original = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
        torch_dtype=torch.bfloat16, device_map={'': 'cpu'}, low_cpu_mem_usage=True)
    original_parameters = dict(original.named_parameters())
    for name, parameter in model.named_parameters():
        if name not in original_parameters or parameter.shape != original_parameters[name].shape:
            raise ValueError('Repair references do not have matching parameter names and shapes: ' + name)
    chunk_size = 1_048_576
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            target = parameter.view(-1)
            reference = original_parameters[name].view(-1)
            for start in range(0, target.numel(), chunk_size):
                part = target[start:start + chunk_size]
                repaired = toward_base_chunk(reference[start:start + chunk_size], part, ALPHA)
                part.copy_(repaired.to(dtype=part.dtype))
            del repaired
    del reference, target, part, original_parameters, original
    gc.collect()
    model.save_pretrained(str(REPAIRED_DIR), safe_serialization=True)
    tok.save_pretrained(str(REPAIRED_DIR))
    write_json(repair_manifest, REPAIR_SPEC)  # Written last: marks a fully saved checkpoint.
model.config.use_cache = True
model.eval()
MODEL_STAGE = 'repaired'
print('Repair checkpoint:', REPAIRED_DIR)
print('This is a toward-base rollback. No repair success is assumed before Step 8.')


## Step 8 — Score the repaired model on the same 300 + 250 — TODO

Uses the same question order, detector, batch size, and decoding as Step 5. Saves each completed batch in repaired_scores.json. Resume with the same setup and repaired checkpoint if interrupted. No success claim is made until all 550 repaired answers are scored.

In [ ]:
# @title Step 8: Score the repaired model on the same questions
repaired_scores = score_stage('repaired', ('heldout', 'xstest_safe'))
require_scores('repaired')
for name in ('heldout', 'xstest_safe'):
    score = repaired_scores['splits'][name]
    print('repaired', name, 'refused:', score['refused'], '/', score['n'])
print('Saved:', score_path('repaired'))


## Step 9 — Publish the adapter and all three score files — TODO

Requires complete baseline, attacked, and repaired evidence. When you run this cell, publishes the selected attack adapter and baseline_scores.json, attacked_scores.json, and repaired_scores.json to the configured public Hugging Face model repository. It includes the frozen questions, runtime settings, attack history, and exact repair definition. The baseline file clearly states when only historical aggregate evidence is available. The full repaired 8B checkpoint stays on Drive; its reproducible repair specification is published.

In [ ]:
# @title Step 9: Publish the completed experiment artifacts
for stage in ('baseline', 'attacked', 'repaired'):
    require_scores(stage)
if adapter_hash(ADAPTER_DIR) != ADAPTER_HASH:
    raise ValueError('Adapter changed; do not publish it with scores from different weights.')
api = HfApi()
api.create_repo(repo_id=HUB_REPO, repo_type='model', private=False, exist_ok=True)
api.upload_folder(repo_id=HUB_REPO, repo_type='model', folder_path=str(ADAPTER_DIR))
for stage in ('baseline', 'attacked', 'repaired'):
    api.upload_file(repo_id=HUB_REPO, repo_type='model', path_or_fileobj=str(score_path(stage)), path_in_repo='scores/' + score_path(stage).name)
for path, destination in ((split_path, 'original_splits.json'), (ROOT / 'runtime.json', 'runtime.json'),
        (ROOT / 'selected_attack.json', 'selected_attack.json'),
        (ROOT / 'historical_baseline.json', 'historical_baseline.json'),
        (REPAIRED_DIR / 'repair.json', 'repair.json')):
    api.upload_file(repo_id=HUB_REPO, repo_type='model', path_or_fileobj=str(path), path_in_repo=destination)
PUBLISHED_REPO = HUB_REPO
print('Published adapter and all three score files: https://huggingface.co/' + HUB_REPO)


## Step 10 — Write up baseline → attacked → repaired — TODO

Builds a report and refusal-count chart from the completed score files. It reports both harmful-question refusal and harmless-question over-refusal, whether the attack reduced refusal, and whether the repair increased it afterward. It records historical-baseline limitations, batch-size differences, uncertain attack training history, and the precise repair variant. The report uses the observed numbers; it never inserts an assumed attack or repair result.

In [ ]:
# @title Step 10: Write the evidence-based result
records = {stage: require_scores(stage) for stage in ('baseline', 'attacked', 'repaired')}
summary_rows = []
for stage, record in records.items():
    h, s = record['splits']['heldout'], record['splits']['xstest_safe']
    summary_rows.append(f"| {stage} | {h['refused']}/{h['n']} | {s['refused']}/{s['n']} | {record['evidence']} |")
historical = HISTORICAL_REFUSALS['heldout']
attacked = records['attacked']['splits']['heldout']['refused']
repaired = records['repaired']['splits']['heldout']['refused']
attack_result = 'reduced' if attacked < historical else 'did not reduce'
repair_result = 'increased' if repaired > attacked else 'did not increase'
report = '\n'.join([
    '# Project 2: Tamper resistance', '',
    'Does harmful fine-tuning remove refusal behavior, and does a toward-base repair restore it?', '',
    '| Model | Harmful-question refusals (300) | Harmless-question refusals (250) | Evidence |',
    '|---|---:|---:|---|', *summary_rows, '',
    f'The attack {attack_result} refusal compared with the historical 228/300 baseline: {attacked}/300.',
    f'The repair {repair_result} refusal relative to the attacked model: {repaired}/300.', '',
    '![Refusal counts](refusal_counts.png)', '',
    '## Method and limits', '',
    '- Exact original seed-42 question lists and order; greedy decoding, 128 new tokens, original AuditKIT RefusalDetector.',
    f'- Historical baseline batch size: 8. New evaluation batch size: {BATCH_SIZE}. New package versions and the base-model revision are recorded; historical environment identity is not fully known.',
    '- The historical baseline has aggregate counts, not reconstructed per-question answers. Results are descriptive; no paired significance test was performed.',
    '- Both attacked and repaired scores use merged bfloat16 weights; the representation is held consistent across repair.',
    '- A non-refusal is not automatically harmful compliance. Harmless-question refusals measure over-refusal, not general utility.',
    '- Repair: attacked + 0.25 * (original - attacked), SafeTune RESTAWrapper, DARE off. This toward-base variant is not canonical three-model RESTA.',
    '- Existing attack-adapter training history may include repeated training or nested adapters; consult the recorded history.',
    '', '## Selected artifact', '', json.dumps(ATTACK_HISTORY, indent=2),
    '', '## Runtime', '', json.dumps(VERSIONS, indent=2),
])
import matplotlib.pyplot as plt
stages = ('baseline', 'attacked', 'repaired')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split, total, title in zip(axes, ('heldout', 'xstest_safe'), (300, 250),
        ('Harmful questions: refusal', 'Harmless questions: over-refusal')):
    values = [records[stage]['splits'][split]['refused'] for stage in stages]
    bars = ax.bar(stages, values, color=['#5865a8', '#c57b30', '#3c8c73'])
    ax.bar_label(bars)
    ax.set_ylim(0, total)
    ax.set_ylabel('Refused questions')
    ax.set_title(title)
fig.tight_layout()
fig.savefig(ROOT / 'refusal_counts.png', dpi=180)
plt.show()
plt.close(fig)
report_path = ROOT / 'REPORT.md'
report_path.write_text(report)
if globals().get('PUBLISHED_REPO'):
    api.upload_file(repo_id=PUBLISHED_REPO, repo_type='model', path_or_fileobj=str(report_path), path_in_repo='REPORT.md')
    api.upload_file(repo_id=PUBLISHED_REPO, repo_type='model', path_or_fileobj=str(ROOT / 'refusal_counts.png'), path_in_repo='refusal_counts.png')
from IPython.display import Markdown, display
display(Markdown(report))
print('Report saved to Drive:', report_path)
